# Question Router: DeBERTa-v3

Train a question-only transformer router for `chartqa`, `docvqa`, and `textvqa`. Default is `microsoft/deberta-v3-small`; switch to `microsoft/deberta-v3-base` if you want a stronger but slower router.

In [ ]:
from pathlib import Path
import json
import random
import subprocess
import sys

REPO_NAME = "multi-task-moe-vlm-assistant"
REPO_URL = "https://github.com/kdnehihi/multi-task-moe-vlm-assistant.git"


def find_project_root(clone_if_missing: bool = True) -> Path:
    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/content") / REPO_NAME,
        Path("/content/drive/MyDrive") / REPO_NAME,
    ]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate

    colab_target = Path("/content") / REPO_NAME
    if clone_if_missing and Path("/content").exists() and not colab_target.exists():
        print(f"Project repo not found. Cloning into {colab_target}...")
        subprocess.run(["git", "clone", REPO_URL, str(colab_target)], check=True)
        if (colab_target / "src").exists():
            return colab_target

    raise ModuleNotFoundError(
        "Could not find project root containing src/. "
        f"Clone {REPO_URL} first, or set PROJECT_ROOT manually."
    )


PROJECT_ROOT = find_project_root(clone_if_missing=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data/processed/router/validation_questions.jsonl"
OUTPUT_DIR = PROJECT_ROOT / "checkpoints/router/deberta_question_router"
MODEL_NAME = "microsoft/deberta-v3-small"  # or "microsoft/deberta-v3-base"
SEED = 42
TEST_SIZE = 0.2
MAX_LENGTH = 96
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
MIN_CONFIDENCE = 0.65
FORCE_REBUILD_ROUTER_DATA = False
ROUTER_DATA_LIMITS = {"docvqa": 1536, "chartqa": 1536, "textvqa": 1536}

random.seed(SEED)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH exists:", DATA_PATH.exists())

## Install Dependencies In Colab If Needed

In [ ]:
# Uncomment in Colab if dependencies are missing.
# %pip install -q datasets transformers accelerate scikit-learn pandas

## Prepare Question-Only Router Data


In [ ]:
# This notebook does not require image files. If gitignored processed data is
# missing, build a small question-only JSONL directly from Hugging Face.

import json
from itertools import islice


def _answer_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return [str(x) for x in value]
    return [str(value)]


def _stream_router_rows(dataset_name: str, limit: int | None, split: str = "validation"):
    from datasets import load_dataset

    if dataset_name == "docvqa":
        dataset = load_dataset("lmms-lab/DocVQA", "DocVQA", split=split, streaming=True)
        for idx, row in enumerate(dataset):
            if limit is not None and idx >= limit:
                break
            yield {
                "dataset": "docvqa",
                "task_type": "docvqa",
                "split": split,
                "question_id": row.get("questionId"),
                "question": row.get("question"),
                "answers": _answer_list(row.get("answers")),
            }
    elif dataset_name == "chartqa":
        hf_split = "val" if split == "validation" else split
        dataset = load_dataset("HuggingFaceM4/ChartQA", split=hf_split, streaming=True)
        for idx, row in enumerate(dataset):
            if limit is not None and idx >= limit:
                break
            yield {
                "dataset": "chartqa",
                "task_type": "chartqa",
                "split": split,
                "question_id": row.get("id") or row.get("id_image"),
                "question": row.get("query") or row.get("question"),
                "answers": _answer_list(row.get("label") or row.get("answer")),
            }
    elif dataset_name == "textvqa":
        dataset = load_dataset("lmms-lab/textvqa", split=split, streaming=True)
        for idx, row in enumerate(dataset):
            if limit is not None and idx >= limit:
                break
            yield {
                "dataset": "textvqa",
                "task_type": "textvqa",
                "split": split,
                "question_id": row.get("question_id"),
                "question": row.get("question"),
                "answers": _answer_list(row.get("answers")),
            }
    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}")


def build_router_question_jsonl(path: Path, limits: dict[str, int | None]) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with path.open("w", encoding="utf-8") as handle:
        for dataset_name in ["docvqa", "chartqa", "textvqa"]:
            dataset_count = 0
            for record in _stream_router_rows(dataset_name, limits.get(dataset_name)):
                question = str(record.get("question") or "").strip()
                if not question:
                    continue
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
                count += 1
                dataset_count += 1
            print(f"{dataset_name}: saved {dataset_count} question records")
    return count


if FORCE_REBUILD_ROUTER_DATA or not DATA_PATH.exists():
    print("Router data missing or rebuild requested. Building question-only data:", DATA_PATH)
    print("Limits:", ROUTER_DATA_LIMITS)
    total = build_router_question_jsonl(DATA_PATH, ROUTER_DATA_LIMITS)
    print("Saved router question records:", total)
else:
    print("Using existing router data:", DATA_PATH)

## Load Questions

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_NAME = globals().get("REPO_NAME", "multi-task-moe-vlm-assistant")
REPO_URL = globals().get("REPO_URL", "https://github.com/kdnehihi/multi-task-moe-vlm-assistant.git")


def find_project_root(clone_if_missing: bool = True) -> Path:
    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/content") / REPO_NAME,
        Path("/content/drive/MyDrive") / REPO_NAME,
    ]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate

    colab_target = Path("/content") / REPO_NAME
    if clone_if_missing and Path("/content").exists() and not colab_target.exists():
        print(f"Project repo not found. Cloning into {colab_target}...")
        subprocess.run(["git", "clone", REPO_URL, str(colab_target)], check=True)
        if (colab_target / "src").exists():
            return colab_target

    raise ModuleNotFoundError(
        "Could not find project root containing src/. "
        f"Clone {REPO_URL} first, or set PROJECT_ROOT manually."
    )


PROJECT_ROOT = globals().get("PROJECT_ROOT")
if PROJECT_ROOT is None or not (Path(PROJECT_ROOT) / "src").exists():
    PROJECT_ROOT = find_project_root(clone_if_missing=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = globals().get("DATA_PATH", PROJECT_ROOT / "data/processed/router/validation_questions.jsonl")

from src.data.answers import canonicalize_task_type

TASK_LABELS = ["chartqa", "docvqa", "textvqa"]
LABEL_TO_ID = {label: i for i, label in enumerate(TASK_LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}


def load_router_records(path: Path) -> list[dict]:
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            question = str(row.get("question", "")).strip()
            if not question:
                continue
            task_type = canonicalize_task_type(row.get("task_type", ""))
            if task_type not in LABEL_TO_ID:
                continue
            records.append({"question": question, "task_type": task_type, "label": LABEL_TO_ID[task_type]})
    return records

records = load_router_records(DATA_PATH)
print("records:", len(records))
print("task counts:", Counter(r["task_type"] for r in records))
records[:3]

## Split

In [ ]:
from sklearn.model_selection import train_test_split

train_records, eval_records = train_test_split(
    records,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=[r["task_type"] for r in records],
)

print("train:", len(train_records), Counter(r["task_type"] for r in train_records))
print("eval:", len(eval_records), Counter(r["task_type"] for r in eval_records))

## Tokenizer And Dataset

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

class QuestionRouterDataset(Dataset):
    def __init__(self, rows, tokenizer, max_length: int = 96):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        encoded = self.tokenizer(
            row["question"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item["labels"] = torch.tensor(row["label"], dtype=torch.long)
        return item


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_dataset = QuestionRouterDataset(train_records, tokenizer, MAX_LENGTH)
eval_dataset = QuestionRouterDataset(eval_records, tokenizer, MAX_LENGTH)

## Train DeBERTa Router

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(TASK_LABELS),
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

args_base = dict(
    output_dir=str(OUTPUT_DIR / "trainer_runs"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    logging_steps=20,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)
try:
    training_args = TrainingArguments(evaluation_strategy="epoch", **args_base)
except TypeError:
    training_args = TrainingArguments(eval_strategy="epoch", **args_base)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate()
metrics

## Evaluation Report

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

pred_output = trainer.predict(eval_dataset)
preds = np.argmax(pred_output.predictions, axis=-1)
true_ids = np.array([r["label"] for r in eval_records])
true_labels = [ID_TO_LABEL[int(i)] for i in true_ids]
pred_labels = [ID_TO_LABEL[int(i)] for i in preds]

print(classification_report(true_labels, pred_labels, labels=TASK_LABELS, digits=4))
cm = confusion_matrix(true_labels, pred_labels, labels=TASK_LABELS)
pd.DataFrame(cm, index=[f"true_{x}" for x in TASK_LABELS], columns=[f"pred_{x}" for x in TASK_LABELS])

## Save Model

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

metadata = {
    "model_type": "deberta_question_router",
    "base_model": MODEL_NAME,
    "output_dir": str(OUTPUT_DIR),
    "data_path": str(DATA_PATH),
    "seed": SEED,
    "test_size": TEST_SIZE,
    "max_length": MAX_LENGTH,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "labels": TASK_LABELS,
    "metrics": {k: float(v) for k, v in metrics.items() if isinstance(v, (int, float))},
    "backend_policy": {
        "chartqa": "chart_dora_r8_a16_B_lr2e-5",
        "docvqa": "base_zero_shot",
        "textvqa": "textvqa_lora_only",
    },
}
(OUTPUT_DIR / "router_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("saved:", OUTPUT_DIR)

## Predict Backend

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_NAME = globals().get("REPO_NAME", "multi-task-moe-vlm-assistant")
REPO_URL = globals().get("REPO_URL", "https://github.com/kdnehihi/multi-task-moe-vlm-assistant.git")


def find_project_root(clone_if_missing: bool = True) -> Path:
    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/content") / REPO_NAME,
        Path("/content/drive/MyDrive") / REPO_NAME,
    ]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate

    colab_target = Path("/content") / REPO_NAME
    if clone_if_missing and Path("/content").exists() and not colab_target.exists():
        print(f"Project repo not found. Cloning into {colab_target}...")
        subprocess.run(["git", "clone", REPO_URL, str(colab_target)], check=True)
        if (colab_target / "src").exists():
            return colab_target

    raise ModuleNotFoundError(
        "Could not find project root containing src/. "
        f"Clone {REPO_URL} first, or set PROJECT_ROOT manually."
    )


PROJECT_ROOT = globals().get("PROJECT_ROOT")
if PROJECT_ROOT is None or not (Path(PROJECT_ROOT) / "src").exists():
    PROJECT_ROOT = find_project_root(clone_if_missing=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = globals().get("DATA_PATH", PROJECT_ROOT / "data/processed/router/validation_questions.jsonl")

from src.routing.task_router import get_backend_for_task, base_fallback_decision


def predict_task_deberta(question: str, min_confidence: float = MIN_CONFIDENCE) -> dict:
    encoded = tokenizer(question, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    encoded = {k: v.to(model.device) for k, v in encoded.items()}
    model.eval()
    with torch.no_grad():
        logits = model(**encoded).logits[0]
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    label_id = int(probs.argmax())
    task_type = ID_TO_LABEL[label_id]
    confidence = float(probs[label_id])
    decision = base_fallback_decision("unknown", confidence) if confidence < min_confidence else get_backend_for_task(task_type, confidence)
    return {
        "task_type": task_type,
        "confidence": confidence,
        "backend_name": decision.backend_name,
        "use_adapter": decision.use_adapter,
        "adapter_name": decision.adapter_name,
        "checkpoint_dir": decision.checkpoint_dir,
    }

predict_task_deberta("What is the highest revenue shown in the chart?")